In [2]:
import pandas as pd

from src.helper import get_split_data

In [3]:
X_trn, y_trn, X_val, y_val, X_tst, y_tst = get_split_data.split_data_for_training(3, 'data/preprocessed/preprocessed_1.csv', 17)

In [4]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV

params = {'colsample_bytree': 0.9010421393996096,
              'gamma': 3.8019677435170545,
              'learning_rate': 0.31296122947271837,
              'max_depth': 10,
              'min_child_weight': 14.140141807395514,
              'n_estimators': 550,
              'reg_alpha': 0.05219808033947845,
              'reg_lambda': 0.33341766540428863,
              'subsample': 0.8566852088545951}

y_trn_value_counts = y_trn.value_counts()
ratio = y_trn_value_counts[0] / y_trn_value_counts[1]

model = XGBClassifier(
    **params,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss",
    scale_pos_weight=ratio
)

calibrated_model = CalibratedClassifierCV(estimator=model, cv=5)
calibrated_model.fit(X_trn, y_trn)

y_pred_proba = calibrated_model.predict_proba(X_tst)

from sklearn.metrics import f1_score
thresholds = np.arange(0.15, 0.9, 0.05)
best_f1 = 0
best_threshold = 0.5

for threshold in thresholds:
    y_pred_threshold = (y_pred_proba[:, 1] >= threshold).astype(int)
    f1 = f1_score(y_val, y_pred_threshold)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Optimal threshold: {best_threshold}, F1: {best_f1}")


Optimal threshold: 0.15, F1: 0.5714285714285714


In [21]:
import featuretools as ft

X = pd.concat([X_trn, X_val], axis=0).reset_index(drop=True)
y = pd.concat([y_trn, y_val], axis=0).reset_index(drop=True)
df = pd.concat([X, y], axis=1).reset_index(drop=True)

es = ft.EntitySet(id='data')
es = es.add_dataframe(dataframe_name='data',
                      dataframe=df,
                      index='match_api_id')

feature_matrix_cleaned = ft.selection.remove_highly_correlated_features(df, pct_corr_threshold=0.85)

/home/kamil/PycharmProjects/predict_football_results/.venv/lib/python3.12/site-packages/featuretools/entityset/entityset.py:1733: UserWarning: index match_api_id not found in dataframe, creating new integer column
  warnings.warn(
